# Lab 05 — Model Comparison for Cyber Threat Detection

## Research question
Which baseline model fails in the least dangerous way?

We compare Logistic Regression, Random Forest, and XGBoost using the same data, split, preprocessing, and metrics. The model is the only thing that changes — that is what makes the comparison fair.

## Learning objectives
- train three baseline models through one shared preprocessing pipeline;
- build one metrics comparison table instead of chasing a single score;
- compare confusion matrices and false-positive/false-negative behavior across models;
- review per-class performance to see which categories stay weak regardless of model;
- write a security interpretation of which model is most useful, and why.


In [ ]:
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

# OPTIONAL — GOOGLE COLAB ONLY: clone the repo so src/ actually exists here
try:
    import google.colab
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()

# If running from the notebooks/ folder locally:
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)


In [ ]:
# OPTIONAL — GOOGLE COLAB ONLY
# Run this cell if the UNSW-NB15 CSV files are not already in DATA_DIR.

try:
    from google.colab import files

    if not (DATA_DIR / "UNSW_NB15_training-set.csv").exists():
        print(
            "Upload UNSW_NB15_training-set.csv, "
            "UNSW_NB15_testing-set.csv, and optionally UNSW_NB15_features.csv"
        )
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)

except ImportError:
    print("Not running in Colab. Put the dataset CSV files in:", DATA_DIR)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline, random_forest_pipeline, xgboost_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame, multiclass_report

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)


## Step 1 — Train all three models through the same pipeline

Only the estimator changes. Data, split, and preprocessing stay identical for each model.

In [ ]:
models = {
    "Logistic Regression": logistic_pipeline(X_train),
    "Random Forest": random_forest_pipeline(X_train),
    "XGBoost": xgboost_pipeline(X_train),
}

comparison_rows = []
predictions = {}
scores = {}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1]

    predictions[name] = pred
    scores[name] = score

    m = binary_metrics(y_test, pred, score)
    comparison_rows.append(binary_metrics_frame(name, m))

comparison_table = pd.concat(comparison_rows)
display(comparison_table.round(4))


## Step 2 — Confusion matrix per model

The comparison table tells you who scored higher. The confusion matrices tell you *how* each model fails — and that is the part that matters for choosing a model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred,
        display_labels=["Normal", "Attack"],
        cmap="Blues", ax=ax, colorbar=False,
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()


## Step 3 — Per-class performance

Overall accuracy or F1 can hide weak performance on the class you actually care about. Compare precision, recall, and F1 for "normal" and "attack" side by side across models.

In [ ]:
per_class_reports = {}

for name, pred in predictions.items():
    report = multiclass_report(y_test, pred)
    report = report.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
    report.index = [f"{name} — {idx}" for idx in report.index]
    per_class_reports[name] = report

display(pd.concat(per_class_reports.values())[["precision", "recall", "f1-score", "support"]].round(3))


## Write your security interpretation

Use this structure:

**Which model has the highest score?**
Name it, and name the metric.

**Which model fails most safely?**
Look at false-negative rate specifically — a model that misses fewer real attacks may be preferable even with a slightly lower F1 or more false positives.

**What would you deploy, and why?**
State your choice and the operational reasoning (missed-attack risk vs. alert-noise risk vs. explainability) — not just the highest number in the table.